# Vector ADD

在本教程中，将使用 Triton 编写一个简单的向量相加 (vector addition) 程序。

同时还将了解：

1. Triton 的基本编程模型
2. 用于定义 Triton 内核的 triton.jit 装饰器 (decorator)
3. 验证和基准测试自定义算子与原生参考实现的最佳实践

In [ ]:
import triton
import triton.language as tl
import torch

DEVICE = triton.runtime.driver.active.get_active_torch_device()

## Compute kernel

In [ ]:
# 向量加的计算核心实现
@triton.jit
def add_kernel(
    x_ptr, # pointer of the input vector x 
    y_ptr, # pointer of the input vector y
    o_ptr, # pointer of the output vector o 
    n_elements, # Size of the vector.
    BLOCK_SIZE: tl.constexpr # Number of elements each program should process.
                             # NOTE: `constexpr` so it can be used as a shape value.
):
    '''
    Compute element-wise addition: o = x + y.
    '''
    # ID of the current Triton program instance
    pid = tl.program_id(0)

    # Integer offsets processed by this program
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # Prevent out-of-bounds memory access
    mask = offs < n_elements

    # Load data from global memory
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)

    # Element-wise addition
    # o = x + y 也可以
    o = tl.add(x, y)

    # Store valid results back to global memory
    tl.store(o_ptr+offs, o, mask=offs<n_elements)

In [ ]:
# 如果想要使用我们用triton写的向量加核心，需要另外定义函数去调用上述核心
def add(x: torch.Tensor, y: torch.Tensor):
    # We need to preallocate the output.
    output = torch.empty_like(x)
    assert x.device == DEVICE and y.device == DEVICE and output.device == DEVICE
    n_elements = output.numel()
    # The SPMD launch grid denotes the number of kernel instances that run in parallel.
    # It is analogous to CUDA launch grids. It can be either Tuple[int], or Callable(metaparameters) -> Tuple[int].
    # In this case, we use a 1D grid where the size is the number of blocks:
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    # NOTE:
    #  - Each torch.tensor object is implicitly converted into a pointer to its first element.
    #  - `triton.jit`'ed functions can be indexed with a launch grid to obtain a callable GPU kernel.
    #  - Don't forget to pass meta-parameters as keywords arguments.
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    # We return a handle to z but, since `torch.cuda.synchronize()` hasn't been called, the kernel is still
    # running asynchronously at this point.
    return output

In [ ]:
# 这样就把计算的核心更换成了我们手写的向量加法核心
# 接下来我们可以与torch的运算结果对比，去验证这个核心的正确性
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
y = torch.rand(size, device=DEVICE)
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

In [ ]:
#更进一步的，我们可以对比torch默认实现和我们用triton实现的运算效率/速度
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],  # Argument names to use as an x-axis for the plot.
        x_vals=[2**i for i in range(12, 28, 1)],  # Different possible values for `x_name`.
        x_log=True,  # x axis is logarithmic.
        line_arg='provider',  # Argument name whose value corresponds to a different line in the plot.
        line_vals=['triton', 'torch'],  # Possible values for `line_arg`.
        line_names=['Triton', 'Torch'],  # Label name for the lines.
        styles=[('blue', '-'), ('green', '-')],  # Line styles.
        ylabel='GB/s',  # Label name for the y-axis.
        plot_name='vector-add-performance',  # Name for the plot. Used also as a file name for saving the plot.
        args={},  # Values for function arguments not in `x_names` and `y_name`.
    ))
def benchmark(size, provider):
    x = torch.rand(size, device=DEVICE, dtype=torch.float32)
    y = torch.rand(size, device=DEVICE, dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]
    if provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
    if provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)
    gbps = lambda ms: 3 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)


benchmark.run(print_data=True, show_plots=True)